# 5차시 실습 · 탐색적 데이터 분석(EDA) — 진짜 데이터 탐색하기
> 데이터분석 6단계 ①~④를 진짜 데이터로 한 바퀴: **질문 → 불러오기·구조 확인 → 정리 → 요약·시각화 → 해석·새 질문**

**진행 방법 (PRIMM)** — 🔮 **예측** → ▶️ **실행** → 👀 **관찰** → 🔧 **수정** → ✏️ **직접 만들기** / ✍️ **해석 쓰기**

**사용 데이터** (`data/` 폴더 — 교재 5장 원본 파일을 이름만 바꿔 복사)
| 파일 | 내용 | 원본 |
|---|---|---|
| `players_2022.csv` | FIFA 22 축구선수 7,772명 × 98열 | 교재 5-1장 `players_2022.csv` (Kaggle) |
| `nhis_treatment_2024_04.csv` | 국민건강보험공단 진료내역정보 2024년 4월 91,732건 × 18열 | 교재 5-2장 (공공데이터포털) |
| `population_2024_04.csv` | 시도별 주민등록인구 2024년 4월 | 교재 5-2장 (행정안전부 주민등록인구통계) |

세 파일 모두 한글 윈도우 인코딩(`cp949`)으로 저장되어 있어 `encoding='cp949'`를 꼭 붙여 읽습니다.

## 0. 준비

도구를 불러오고 한글 폰트를 설정한다. (코랩에서는 `koreanize-matplotlib`가 자동 설치된다)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import koreanize_matplotlib
except ModuleNotFoundError:
    %pip install -q koreanize-matplotlib
    import koreanize_matplotlib

plt.rcParams['figure.figsize'] = (8, 4)
DATA = 'data' if os.path.exists('data') else '../data'
print('데이터 폴더:', DATA)

---
# PART A. FIFA 축구선수 데이터 탐색
## ① 문제 정의 — 무엇이 궁금한가?
- **Q1.** 선수들의 종합 능력치(`overall`)는 어떻게 분포할까?
- **Q2.** 어느 나라 선수가 가장 많을까?
- **Q3.** 최상위 100명은 어느 클럽에 몰려 있을까?
- **Q4.** 한국 선수들은 어떤 특징이 있을까?
- **Q5.** 나이와 능력치는 어떤 관계일까? 10대 유망주는 누구일까?
- **Q6.** 손흥민 선수는 다른 선수들과 비교해 얼마나 뛰어날까?

## ② 불러오기와 구조 확인

▶️ CSV를 `cp949` 인코딩으로 불러와 크기를 확인한다. (인코딩을 빼면 `UnicodeDecodeError`가 난다)

In [ ]:
df = pd.read_csv(f'{DATA}/players_2022.csv', encoding='cp949')
print(df.shape)
df.head(3)

👀 관찰: 열이 98개나 된다. 열 이름을 모두 훑어본다.

In [ ]:
print(df.columns.tolist())

✏️ 빈칸 채우기: 빈 값(결측치)이 있는 열만 골라 개수를 본다.

In [ ]:
na = df.isna().___
na[na ___].head(15)

## ③ 정리 — 질문에 필요한 열만 고르기

▶️ 교재는 `df.iloc[:, [1,3,7,11,14,18,30,31,32,33,34,35]]`처럼 **번호**로 열을 골랐다. 같은 열을 **이름**으로 고르면 무엇을 골랐는지 읽기 쉽다.

In [ ]:
cols = ['short_name', 'overall', 'age', 'club_name', 'club_position', 'nationality_name',
        'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
df_fifa = df[cols].copy()
print(df_fifa.shape)
df_fifa.describe().round(1)

🔮 예측 → 실행: 능력치(pace~physic)가 709개씩 비어 있다. 어떤 선수들일까? (예측: ___ )

In [ ]:
df_fifa[df_fifa['pace'].isna()]['club_position'].value_counts(dropna=False)

👀 **관찰** — 비어 있는 선수는 대부분 골키퍼(GK)다. 골키퍼는 필드 선수용 능력치가 없어서 빈칸으로 둔 것이다.
**결측치는 '오류'가 아니라 '의미'일 수 있다.** 지우기 전에 먼저 누구인지 확인하자.

👀 관찰: 데이터 품질 점검 — 선수·클럽 이름에 `?`가 들어간 경우가 있다. (예: `FC Bayern M?nchen`)

In [ ]:
broken = df_fifa['club_name'].str.contains('?', regex=False, na=False)
print('이름이 깨진 클럽 행 수:', broken.sum())
df_fifa[broken]['club_name'].value_counts().head(5)

➡️ 원본 파일을 `cp949`로 저장하면서 ü, é 같은 글자가 `?`로 바뀌었다. 클럽 이름으로 집계할 때 결과에 `?`가 보여도 같은 클럽이다.

## ④ 요약·시각화

### Q1. 종합 능력치(overall)의 분포

✏️ 빈칸 채우기: overall의 평균·중앙값을 구하고 히스토그램을 그린다.

In [ ]:
print('선수 수:', len(df_fifa))
print('평균:', round(df_fifa['overall'].___, 1), '중앙값:', df_fifa['overall'].___)
plt.hist(df_fifa['overall'], bins=20, edgecolor='white')
plt.title('선수 종합 능력치(overall) 분포')
plt.xlabel('overall')
plt.ylabel('선수 수')
plt.show()

▶️ overall 상위 10명을 본다. (같은 점수끼리는 실행 환경에 따라 순서가 달라질 수 있다)

In [ ]:
df_fifa.sort_values(by='overall', ascending=False).head(10)[['short_name', 'overall', 'age', 'club_name', 'nationality_name']]

### Q2. 국적별 선수 수 TOP 10

✏️ 빈칸 채우기: 국적별 선수 수를 세서 상위 10개를 막대 그래프로 그린다.

In [ ]:
nation_top10 = df_fifa['nationality_name'].___.head(10)
nation_top10.plot.bar(rot=45)
plt.title('국적별 선수 수 TOP 10')
plt.xlabel('국적')
plt.ylabel('선수 수')
plt.show()

### Q3. 랭킹 TOP 100 선수의 소속 클럽

✏️ 빈칸 채우기: overall 상위 100명을 뽑고, 클럽별 인원 상위 10개를 가로 막대로 그린다. (`invert_yaxis`로 1위를 맨 위로)

In [ ]:
top100 = df_fifa.sort_values(by='overall', ascending=False).___
top100['club_name'].value_counts().head(10).plot.___()
plt.gca().invert_yaxis()
plt.title('랭킹 TOP 100 선수의 클럽 분포')
plt.xlabel('선수 수')
plt.ylabel('클럽')
plt.show()

### Q4. 한국 선수 분석

✏️ 빈칸 채우기: 국적이 'Korea Republic'인 선수만 걸러내고(2차시 조건 필터), overall이 높은 순으로 10명을 본다.

In [ ]:
df_korea = df_fifa[df_fifa['nationality_name'] == 'Korea Republic']
print('한국 선수 수:', len(df_korea))
print('한국 선수 평균 overall:', round(df_korea['overall'].mean(), 1), '/ 전체 평균:', round(df_fifa['overall'].mean(), 1))
df_korea.sort_values(by='overall', ascending=___).head(10)[['short_name', 'overall', 'club_name', 'club_position']]

▶️ 한국 선수 상위 50명의 소속 클럽 TOP 10

In [ ]:
korea_top50 = df_korea.sort_values(by='overall', ascending=False).head(50)
korea_top50['club_name'].value_counts().head(10).plot.barh(color='crimson')
plt.gca().invert_yaxis()
plt.title('한국 선수 TOP 50의 클럽 분포')
plt.xlabel('선수 수')
plt.show()

✏️ 빈칸 채우기: 한국 선수 TOP 50의 포지션 구성을 원 그래프로, 포지션별 평균 overall을 groupby로 구한다.

In [ ]:
position = korea_top50['club_position'].value_counts()
plt.figure(figsize=(4, 4))
plt.pie(position.values, labels=position.index, autopct=___, startangle=90, counterclock=False)
plt.title('한국 선수 TOP 50 포지션 구성')
plt.axis('equal')
plt.show()
korea_top50.___['overall'].mean().round(1).sort_values(ascending=False)

### Q5. 나이와 능력치 — 10대 유망주

✏️ 빈칸 채우기: 20세 미만 선수 수와 상위 5명을 본다.

In [ ]:
teen = df_fifa[df_fifa['age'] ___]
print('20세 미만 선수:', len(teen))
teen.sort_values(by='overall', ascending=False).head(5)[['short_name', 'age', 'overall', 'club_name']]

▶️ 나이와 overall의 산점도, 그리고 나이별 평균 overall 선 그래프

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(df_fifa['age'], df_fifa['overall'], alpha=0.2, s=8, color='orange')
axes[0].set_title('나이 vs overall (선수 한 명 = 점 하나)')
axes[0].set_xlabel('나이')
axes[0].set_ylabel('overall')
age_mean = df_fifa.groupby('age')['overall'].mean()
axes[1].plot(age_mean.index, age_mean.values, marker='o')
axes[1].set_title('나이별 평균 overall')
axes[1].set_xlabel('나이')
plt.tight_layout()
plt.show()
df_fifa['age'].value_counts().sort_index().tail(6)  # 나이 많은 쪽 인원수 확인

✍️ **해석 쓰기** — 나이별 평균 overall은 몇 살쯤까지 오르다가 멈추나? 40세 이상에서 선이 크게 흔들리는 이유를 인원수와 함께 설명해 보자.

> (여기에 한두 문장으로 적어 보세요)

### Q6. 관심 선수 심화 분석 — 손흥민

▶️ 한국 국적이면서 이름에 'Son'이 들어간 선수를 찾는다. (`str.contains`)

In [ ]:
df_fifa[(df_fifa['nationality_name'] == 'Korea Republic') & (df_fifa['short_name'].str.contains('Son'))]

✏️ 빈칸 채우기: 손흥민(H. Son)을 전체 평균, 같은 포지션 평균과 비교하고, 상위 몇 %인지 구한다.

In [ ]:
son = df_fifa[df_fifa['short_name'] == 'H. Son'].iloc[0]
same_position = df_fifa[df_fifa['club_position'] == son['club_position']]
print(f"손흥민 overall: {son['overall']} (포지션 {son['club_position']})")
print(f"전체 평균: {df_fifa['overall'].___:.2f}")
print(f"같은 포지션 평균: {same_position['overall'].mean():.2f}")
top_pct = (df_fifa['overall'] ___ son['overall']).mean() * 100
print(f"손흥민보다 overall이 높은 선수 비율: {top_pct:.2f}%")

✍️ **해석 쓰기** — 손흥민 선수의 능력치를 '숫자 → 비교 → 의미와 한계' 3단계로 설명해 보자. (FIFA 게임 능력치라는 점도 한계에 넣자)

> (여기에 한두 문장으로 적어 보세요)

---
# PART B. 국민건강보험공단 진료내역 데이터 탐색
**데이터 이해** — 국민건강보험공단이 가입자 중 **100만 명을 무작위로 뽑아(표본)** 진료 기록을 공개한 자료 중 2024년 4월 일부다.
이름·주민번호 같은 개인식별정보는 빠져 있고, 가입자는 일련번호로만 구분되며 85세 이상은 '85세+'로 묶여 있다(비식별화).

## ① 문제 정의
- **Q1.** 한 달 동안 진료는 몇 건이고, 한 사람이 평균 몇 번 진료받았을까? 가장 많은 병은?
- **Q2.** 어느 연령대가 진료를 많이 받을까?
- **Q3.** 어느 지역 사람들이 진료를 많이 받을까? (인구 차이를 고려하면?)
- **Q4.** 나이가 많을수록 처방일수도 길어질까?
- **Q5.** 남녀는 주로 어떤 진료과목을 이용할까?

## ② 불러오기와 구조 확인

▶️ 진료내역 데이터를 불러와 구조를 확인한다.

In [ ]:
raw = pd.read_csv(f'{DATA}/nhis_treatment_2024_04.csv', encoding='cp949')
print(raw.shape)
raw.info()

✏️ 빈칸 채우기: 질문에 필요한 9개 열만 이름으로 고르고, 결측치 개수를 센다. (교재: `iloc[:, [0, 1, 3, 4, 5, 6, 8, 9, 17]]`과 같다)

In [ ]:
cols = ['기준년도', '가입자일련번호', '성별코드', '연령대코드', '시도코드',
        '요양개시일자', '진료과목코드', '주상병코드', '총처방일수']
med = raw[cols].copy()
print(med.isna().___.sum(), '개의 결측치')
med.head()

## ③ 정리 — 코드를 사람이 읽는 이름으로 (파생 변수)

✏️ 빈칸 채우기: 성별코드(1·2)를 `replace`와 딕셔너리로 '남'·'여'로 바꾼 '성별' 열을 만들고, 성별 건수를 센다.

In [ ]:
med['성별'] = med['성별코드'].replace({1: '남', 2: '여'})
med['성별'].___

▶️ 연령대코드·시도코드·진료과목코드도 사용자 매뉴얼의 표대로 이름을 붙인다.

In [ ]:
age_dict = {1: '0~4세', 2: '5~9세', 3: '10~14세', 4: '15~19세', 5: '20~24세', 6: '25~29세',
            7: '30~34세', 8: '35~39세', 9: '40~44세', 10: '45~49세', 11: '50~54세', 12: '55~59세',
            13: '60~64세', 14: '65~69세', 15: '70~74세', 16: '75~79세', 17: '80~84세', 18: '85세+'}
city_dict = {11: '서울특별시', 26: '부산광역시', 27: '대구광역시', 28: '인천광역시', 29: '광주광역시',
             30: '대전광역시', 31: '울산광역시', 36: '세종특별자치시', 41: '경기도', 42: '강원특별자치도',
             43: '충청북도', 44: '충청남도', 45: '전북특별자치도', 46: '전라남도', 47: '경상북도',
             48: '경상남도', 49: '제주특별자치도'}
clinic_dict = {0: '일반의', 1: '내과', 2: '신경과', 3: '정신과', 4: '외과', 5: '정형외과', 6: '신경외과',
               7: '흉부외과', 8: '성형외과', 9: '마취통증의학과', 10: '산부인과', 11: '소아청소년과',
               12: '안과', 13: '이비인후과', 14: '피부과', 15: '비뇨기과', 16: '영상의학과',
               17: '방사선종양학과', 18: '병리과', 19: '진단검사의학과', 20: '결핵과', 21: '재활의학과',
               22: '핵의학과', 23: '가정의학과', 24: '응급의학과', 25: '산업의학과', 26: '예방의학과'}
med['연령대'] = med['연령대코드'].replace(age_dict)
med['시도'] = med['시도코드'].replace(city_dict)
med['진료과목'] = med['진료과목코드'].replace(clinic_dict)
order = list(age_dict.values())   # 연령대 순서 (그래프에서 사용)
med.head()

## ④ 요약·시각화

### Q1. 진료건수·환자 수·많은 병

✏️ 빈칸 채우기: 전체 진료건수, 서로 다른 가입자 수, 1인당 평균 진료건수를 구한다.

In [ ]:
n_visit = len(med)
n_person = med['가입자일련번호'].___
print('진료건수:', n_visit, '/ 가입자 수:', n_person, '/ 1인당:', round(n_visit / n_person, 2))

▶️ 가장 많이 나온 주상병코드 TOP 3와 진료과목 TOP 3

In [ ]:
print(med['주상병코드'].value_counts().head(3))
# J209: 급성 기관지염 · I109: 고혈압 · J304: 알레르기비염 (한국표준질병사인분류)
print(med['진료과목'].value_counts().head(3))

✍️ **해석 쓰기** — 4월에 급성 기관지염과 알레르기비염 진료가 많은 이유를 추측해 보자. 이 추측을 확인하려면 어떤 데이터가 더 필요할까?

> (여기에 한두 문장으로 적어 보세요)

### Q2. 연령대별 진료건수

✏️ 빈칸 채우기: 연령대 순서를 지정해 countplot을 그린다.

In [ ]:
sns.countplot(data=med, x=___, order=order, color='teal')
plt.title('연령대별 진료건수 (2024년 4월)')
plt.xticks(rotation=45)
plt.ylabel('진료건수')
plt.show()

👀 관찰: 그래프에 0~4세 막대가 없다. 정말 없는지 숫자로 확인한다.

In [ ]:
med['연령대코드'].value_counts().sort_index().head(3)

➡️ 이 파일에는 연령대코드 1(0~4세)이 한 건도 없다. 원래 없는 것인지, 공개 과정에서 빠진 것인지는 데이터만으로 알 수 없다 — **해석할 때 한계로 꼭 적자.**

### Q3. 시도별 진료건수 — 인구를 고려하면?

▶️ 시도별 진료건수를 센다.

In [ ]:
city_cnt = med['시도'].value_counts()
sns.countplot(data=med, x='시도', order=city_cnt.index, color='steelblue')
plt.title('시도별 진료건수')
plt.xticks(rotation=60)
plt.show()

🔮 예측: 경기도가 가장 많다. 경기도 사람들이 병원에 가장 자주 간다고 말할 수 있을까? (예측: ___ ) — 인구 데이터를 붙여 확인한다.

In [ ]:
pop = pd.read_csv(f'{DATA}/population_2024_04.csv', encoding='cp949')
city_pop = pop[['행정구역(동읍면)별', '계']].drop(0)        # 0번 행 = 전국 합계
city_pop.columns = ['시도', '인구수']
df_city = city_cnt.reset_index().rename(columns={'count': '진료건수'})
df_city = pd.merge(df_city, city_pop, on='시도', how='inner')
df_city.head()

✏️ 빈칸 채우기: 인구 1만 명당 진료건수를 계산하고 정렬해 그린다.

In [ ]:
df_city['1만명당_진료건수'] = df_city['진료건수'] ___ df_city['인구수'] * 10000
df_city = df_city.sort_values(by=___, ascending=False)
sns.barplot(data=df_city, x='시도', y='1만명당_진료건수', color='darkorange')
plt.title('시도별 인구 1만 명당 진료건수 (표본 기준 상대 비교)')
plt.xticks(rotation=60)
plt.show()
df_city[['시도', '진료건수', '1만명당_진료건수']].round(1).head(5)

✍️ **해석 쓰기** — 진료건수 1위였던 경기도는 인구 대비로 보면 몇 위가 되었나? '전체 개수'와 '인구 대비 비율' 중 무엇으로 비교해야 공정할까?

> (여기에 한두 문장으로 적어 보세요)

### Q4. 연령대와 처방일수

▶️ 수치형 열의 상관계수를 히트맵으로 본다. (교재 5-2장)

In [ ]:
corr = med.drop(columns='기준년도').corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Reds')
plt.title('수치형 열 상관계수')
plt.show()

👀 **관찰** — `성별코드`·`시도코드`·`진료과목코드`는 숫자로 적혀 있지만 **범주형**이다(3차시). 시도코드 41이 11보다 '크다'는 뜻이 없으므로 이 열들의 상관계수는 의미가 없다.
의미 있게 읽을 수 있는 것은 **순서가 있는 `연령대코드`와 `총처방일수`(0.39)** 정도다.

✏️ 빈칸 채우기: 연령대별 평균 총처방일수를 막대로, 성별로 나눠 그린다.

In [ ]:
plt.figure(figsize=(11, 4))
sns.barplot(data=med, x='연령대', y='총처방일수', hue=___, order=order[1:], errorbar=None)
plt.title('연령대·성별 평균 총처방일수')
plt.xticks(rotation=45)
plt.ylabel('평균 총처방일수(일)')
plt.show()

### Q5. 성별 진료과목 TOP 3

✏️ 빈칸 채우기: 성별·진료과목별 진료건수를 세고, 성별마다 상위 3개를 본다.

In [ ]:
clinic = med.groupby([___]).size().reset_index(name='진료건수')
for g in ['여', '남']:
    print(g, clinic[clinic['성별'] == g].sort_values('진료건수', ascending=False).head(3)[['진료과목', '진료건수']].values.tolist())

---
# PART C. ✏️ 직접 탐색하기 — 나만의 질문 하나
두 데이터 중 하나를 골라 **질문 → 필요한 열 → 요약·그래프 → 해석·새 질문**을 한 바퀴 돌려 보자.
아래는 예시 두 개다. 그대로 따라 해도 되고, 나만의 질문으로 바꿔도 된다.

C1. (FIFA 예시) 포지션별로 속도(pace) 분포는 어떻게 다를까? — 골키퍼는 pace가 없으니 자동으로 빠진다.

In [ ]:
sns.boxplot(data=df_fifa, x=___, y='pace', order=['FW', 'MF', 'DF'])
plt.title('포지션별 속도(pace) 분포')
plt.show()

C2. (진료내역 예시) 요일에 따라 진료건수가 다를까? — 날짜 열에서 요일을 뽑아낸다(새 파생 변수).

In [ ]:
med['요일'] = pd.to_datetime(med['요양개시일자']).dt.___.map({0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'})
sns.countplot(data=med, x='요일', order=['월', '화', '수', '목', '금', '토', '일'], color='seagreen')
plt.title('요일별 진료건수 (2024년 4월)')
plt.show()

👀 관찰: 날짜별로 보면 더 자세히 보인다. 일별 진료건수를 선 그래프로 그린다.

In [ ]:
daily = med['요양개시일자'].value_counts().sort_index()
daily.plot(marker='o', rot=60, figsize=(10, 3.5), title='일별 진료건수 (2024년 4월)')
plt.ylabel('진료건수')
plt.show()
daily.loc[['2024-04-09', '2024-04-10', '2024-04-11']]

➡️ 주말(토·일)은 거의 0건이고, **4월 10일(수)도 2건뿐**이다. 2024년 4월 10일은 국회의원 선거일로 공휴일이었다.
또 4월에는 수요일이 네 번(월·화는 다섯 번)뿐이라 요일 합계를 그대로 비교하면 불공평하다. **그래프에서 '이상한 점'을 찾으면 → 이유를 찾고 → 한계로 적는다.** 이것이 EDA다.
(주말이 거의 없는 이유는 이 파일이 원본에서 어떻게 추려졌는지 알아야 확인할 수 있다.)

C3. 나만의 질문을 정하고 코드로 탐색한다.

In [ ]:
# 나의 질문:
___

### 📋 EDA 보고 카드 (직접 채우기)
| 항목 | 내용 |
|---|---|
| 나의 질문 | |
| 사용한 데이터·열 | |
| 사용한 방법·그래프 | |
| 발견한 것 (숫자 → 비교) | |
| 한계 (표본·결측·인코딩·기간 등) | |
| 새로 생긴 질문 | |

---
### 오늘의 정리 — EDA 한 바퀴
1. **질문**을 먼저 적는다 → 2. 불러와 **구조**(shape·info·isna)를 본다 → 3. 필요한 열만 고르고 **코드를 이름으로** 바꾼다
→ 4. **세고(value_counts)·나누고(groupby)·그린다**(4차시) → 5. **해석하고, 한계를 적고, 새 질문**을 만든다
- 결측치는 오류가 아니라 의미일 수 있다 (골키퍼의 빈 능력치)
- 전체 개수보다 **비율**(인구 1만 명당)이 공정한 비교일 때가 많다
- 숫자로 적힌 범주형 코드의 상관계수는 의미가 없다